In [ ]:
# imports
import torch
import shutil
import numpy as np
from torch import nn
from pathlib import Path
from collections import OrderedDict
from kagglehub import dataset_download
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from torchvision import transforms, datasets
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset, DataLoader, WeightedRandomSampler
from sklearn.metrics import f1_score, accuracy_score
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

In [ ]:
# download dataset

local_data_path = Path("../data")
dataset_base_path = Path(dataset_download("crawford/cat-dataset"))
dataset_base_path = dataset_base_path / "cats"
oreo_path = local_data_path / "oreo"
not_oreo_path = local_data_path / "not_oreo"

oreo_path.mkdir(exist_ok=True)
not_oreo_path.mkdir(exist_ok=True)

for image in local_data_path.glob("*.jpg"):
    shutil.move(str(image), str(oreo_path / image.name))

for subdir in dataset_base_path.iterdir():
    if subdir.is_dir():
        for image in subdir.glob("*.jpg"):
            new_name = f"{subdir.name}_{image.name}"
            shutil.move(str(image), str(not_oreo_path / new_name))

In [ ]:
# preprocessing

# Resize, Augment, and Normalize RGB
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    #transforms.RandomHorizontalFlip(p=0.5),
    #transforms.RandomRotation(degrees=15),
    #transforms.ColorJitter(brightness=0.2), # random brightness
    transforms.ToTensor(),
])

full_dataset = datasets.ImageFolder(root=local_data_path, transform=transform)
train_loader = DataLoader(full_dataset, batch_size=32, shuffle=True)

In [ ]:
# Split

targets = np.array(full_dataset.targets)
indices = np.arange(len(full_dataset))

# 80% Train
train_idx, temp_idx = train_test_split(
    indices,
    test_size=0.2,
    stratify=targets,
    random_state=42
)

# 10% validation
# 10% test
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.5,
    stratify=targets[temp_idx],
    random_state=42
)

train_dataset = Subset(full_dataset, train_idx)
val_dataset = Subset(full_dataset, val_idx)
test_dataset = Subset(full_dataset, test_idx)

In [ ]:
# Weighted Random Sampler

train_targets = targets[train_idx]
class_sample_count = np.array([len(np.where(train_targets == t)[0]) for t in np.unique(train_targets)])
weight = 1. / class_sample_count
samples_weight = np.array([weight[t] for t in train_targets])
samples_weight = torch.from_numpy(samples_weight)

sampler = WeightedRandomSampler(
    weights=samples_weight,
    num_samples=len(samples_weight),
    replacement=True
)

train_loader = DataLoader(train_dataset, batch_size=32, sampler=sampler, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4)

images, labels = next(iter(train_loader))
oreo_count = (labels == full_dataset.class_to_idx['oreo']).sum().item()
not_oreo_count = (labels == full_dataset.class_to_idx['not_oreo']).sum().item()

In [ ]:
# flatten images, convert to numpy

# train
x_train = []
y_train = []
for images, labels in train_loader:
    x_train.append(images.view(images.size(0), -1).numpy())
    y_train.append(labels.numpy())
x_train = np.vstack(x_train)
y_train = np.concatenate(y_train)

# validation
x_val = []
y_val = []
for images, labels in val_loader:
    x_val.append(images.view(images.size(0), -1).numpy())
    y_val.append(labels.numpy())
x_val = np.vstack(x_val)
y_val = np.concatenate(y_val)

# test
x_test = []
y_test = []
for images, labels in test_loader:
    x_test.append(images.view(images.size(0), -1).numpy())
    y_test.append(labels.numpy())
x_test = np.vstack(x_test)
y_test = np.concatenate(y_test)

In [ ]:
# logistic Regression
pca = PCA(n_components=0.95)
x_train = pca.fit_transform(x_train)
x_val = pca.transform(x_val)
x_test = pca.transform(x_test)

# hyperparameter tuning
C_values = [0.01, 0.1, 1]
best_regression = None
best_f1 = 0
best_C = None

# f1 validation
for C in C_values:
    regression = LogisticRegression(max_iter=3000, solver='saga', C=C, class_weight='balanced')

    regression.fit(x_train, y_train)
    val_preds = regression.predict(x_val)

    f1 = f1_score(y_val, val_preds, average='macro')
    accuracy = accuracy_score(y_val, val_preds)

    print(f"C={C}, Validation F1={f1}, Validation Accuracy: {accuracy}")
    if f1 > best_f1:
        best_f1 = f1
        best_regression = regression
        best_C = C

print(f"\nBest C: {best_C} with F1={best_f1}")

predictions = best_regression.predict(x_test)
print(f"Predictions: {predictions}")
print(f"Actual labels: {y_test}")
print(f"Accuracy: {accuracy_score(y_test, predictions)}")

In [ ]:
# K-NN
# Comment out all but resizing preprocessing before running
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, shuffle=False)
val_loader = DataLoader(val_dataset, shuffle=False)

# Creating arrays for training
x_train = []
y_train = []
for images, labels in train_loader:
    # Flattening images and converting to numpy
    x_train.append(images.view(images.size(0), -1).numpy())
    y_train.append(labels.numpy())
x_train = np.vstack(x_train)
y_train = np.concatenate(y_train)

# Creating arrays for validation
x_val = []
y_val = []
for images, labels in val_loader:
    x_val.append(images.view(images.size(0), -1).numpy())
    y_val.append(labels.numpy())
x_val = np.vstack(x_val)
y_val = np.concatenate(y_val)

# Validation
k_val = [1, 2, 3, 5, 7, 9]
best_k = None
best_f1 = 0
for k in k_val:
    model = KNeighborsClassifier(n_neighbors=k, n_jobs=-1)
    model.fit(x_train, y_train)
    val_predictions = model.predict(x_val)
    f1 = f1_score(y_val, val_predictions, average='macro')
    print(f"k={k}, Validation F1 Score: {f1:.4f}")
    if f1 > best_f1:
        best_f1 = f1
        best_k = k

# Training with best k
print(f"Best k: {best_k} with F1 Score: {best_f1:.4f}")
neighbors = KNeighborsClassifier(n_neighbors=best_k, n_jobs=-1)
neighbors.fit(x_train, y_train)

# Testing
# Creating arrays for testing
x_test = []
y_test = []
for images, labels in test_loader:
    x_test.append(images.view(images.size(0), -1).numpy())
    y_test.append(labels.numpy())
x_test = np.vstack(x_test)
y_test = np.concatenate(y_test)

# Printing predictions and actual labels
predictions = neighbors.predict(x_test)
print(f"Predictions: {predictions}")
print(f"Actual labels: {y_test}")

accuracy = np.mean(predictions == y_test)
print(f"Test Accuracy: {accuracy:.4f}")

# Showing incorrect predictions and their neighbors
incorrect = np.where(((predictions == 1) & (y_test == 0)) | ((predictions == 0) & (y_test == 1)))[0] 
num_mistakes = len(incorrect) 
if num_mistakes > 0: 
    fig, axes = plt.subplots(num_mistakes, best_k + 1, figsize=(20, 4 * num_mistakes)) 
   
    if num_mistakes == 1: 
        axes = np.expand_dims(axes, axis=0) 
    for i, test_idx in enumerate(incorrect): 
        sample_image = x_test[test_idx]
        distances,  neighbor_indices = neighbors.kneighbors(sample_image.reshape(1, -1), n_neighbors=best_k)
        ax_test = axes[i, 0]
        img_reshaped = sample_image.reshape(3, 128, 128).transpose(1, 2, 0) 
        ax_test.imshow(img_reshaped) 
        ax_test.set_title(f"INCORRECT\n(Index {test_idx})", color='red')  
        ax_test.axis('off')
        for j, train_idx in enumerate(neighbor_indices[0]): 
            ax_nb = axes[i, j + 1] 
            neighbor_img = x_train[train_idx].reshape(3, 128, 128).transpose(1, 2, 0)
            ax_nb.imshow(neighbor_img) 
            ax_nb.set_title(f"Neighbor {j+1}\nDist: {distances[0][j]:.2f}") 
            ax_nb.axis('off')

plt.tight_layout() 
plt.show()


In [ ]:
# CNN
learning_rate = 0.01
batch_size = 32
epochs = 2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

cnn_model = nn.Sequential(
    OrderedDict([
        ("conv1", nn.Conv2d(3, 128, kernel_size=3, padding=1)),
        ("relu1", nn.ReLU()),
        ("pool1", nn.MaxPool2d(kernel_size=2, stride=2)),
        ("conv2", nn.Conv2d(128, 64, kernel_size=3, padding=1)),
        ("relu2", nn.ReLU()),
        ("pool2", nn.MaxPool2d(kernel_size=2, stride=2)),
        ("conv3", nn.Conv2d(64, 32, kernel_size=3, padding=1)),
        ("relu3", nn.ReLU()),
        ("pool3", nn.MaxPool2d(kernel_size=2, stride=2)),
        ("flatten", nn.Flatten()),
        ("linear1", nn.Linear(32 * 16 * 16, 128)),
        ("relu5", nn.ReLU()),
        ("linear2", nn.Linear(in_features=128, out_features=1)),
    ])
).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(cnn_model.parameters(), lr=learning_rate)

cnn_model.train()
for epoch in range(epochs):
    total_loss = 0

    for i, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.float().to(device)

        optimizer.zero_grad()

        output = cnn_model(images).squeeze(1)
        loss = criterion(output, labels)
        total_loss += loss.item()

        loss.backward()
        optimizer.step()

    print(f"epoch {epoch + 1}/{epochs}")
    print(f"epoch {total_loss:.2f}")

In [ ]:
# evaluation